# NB08 — Revenue Impact Estimation

We estimate the revenue hospitals could recover through better documentation. The formula:

$$\text{revenue\_opportunity} = \text{discharges} \times \text{estimated\_shift\_rate} \times \text{avg\_payment\_uplift}$$

This is a **conservative estimate** — actual CDI (Clinical Documentation Improvement) programs typically capture more. We calculate the potential revenue uplift if hospitals improved documentation to shift cases to higher severity tiers (CC or MCC).

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Define paths
gap_scores_path = '../../data/outputs/nb07_gap_scores/hospital_gap_scores.csv'
severity_path = '../../data/outputs/nb02_drg_cmi/severity_tier_payments.csv'
output_path = '../../data/outputs/nb08_revenue_impact/hospital_revenue_impact.csv'

# Load data
print('Loading hospital gap scores...')
df = pd.read_csv(gap_scores_path, dtype={'ccn': str})
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')

print('\nLoading severity tier payment data...')
df_sev = pd.read_csv(severity_path)
print(f'Shape: {df_sev.shape}')
print(f'Columns: {df_sev.columns.tolist()}')

Loading hospital gap scores...
Shape: (3280, 26)
Columns: ['ccn', 'hospital_name', 'state', 'beds', 'bed_size_tier', 'ownership_category', 'is_teaching', 'census_region', 'is_urban', 'cmi', 'peer_cmi_mean', 'cmi_gap', 'cmi_z_score', 'mcc_ratio', 'peer_mcc_ratio_mean', 'mcc_gap', 'avg_payment_per_discharge', 'payment_gap', 'beds.1', 'drg_diversity', 'cmi_gap_score', 'severity_score', 'payment_score', 'complexity_score', 'doc_gap_score', 'gap_tier']

Loading severity tier payment data...
Shape: (313, 9)
Columns: ['drg_family', 'with_CC', 'with_MCC', 'without_CC', 'payment_without_CC', 'payment_with_CC', 'payment_with_MCC', 'cc_uplift', 'mcc_uplift']


## Revenue Impact Model

### Methodology

**Severity Shift Rate**: Estimated percentage of cases that could be reclassified with better documentation. Based on the hospital's Documentation Gap Score — higher gap indicates more opportunities for improvement. **These rates are analyst-defined assumptions, not empirically calibrated from chart-audit data.** They are conservative relative to published CDI benchmarks (15–25% improvement rates).

**Important**: The shift rate tiers are aligned with the composite Documentation Gap Score tiers (Low 0–49, Medium 50–64, High 65–100) to maintain consistency across the analysis.

**Payment Uplift**: We calculate two key metrics:
- **CC Uplift**: Average payment increase when shifting to CC (complications/comorbidities) tier
- **MCC Uplift**: Average payment increase when shifting to MCC (major complications/comorbidities) tier

**Blended Approach**: We assume a realistic mix of shifts:
- 70% of improved cases shift to CC tier
- 30% of improved cases shift to MCC tier

This conservative model avoids assuming all cases could reach the highest severity tier.

In [2]:
# Calculate average uplift values from severity tier data
avg_cc_uplift = df_sev['cc_uplift'].dropna().mean()
avg_mcc_uplift = df_sev['mcc_uplift'].dropna().mean()

# Blended uplift: 70% CC, 30% MCC
blended_uplift = (avg_cc_uplift * 0.70) + (avg_mcc_uplift * 0.30)

print('Payment Uplift Analysis')
print('='*50)
print(f'Average CC uplift: ${avg_cc_uplift:,.0f}')
print(f'Average MCC uplift: ${avg_mcc_uplift:,.0f}')
print(f'Blended uplift (70% CC / 30% MCC): ${blended_uplift:,.0f}')
print(f'\nBased on {df_sev.shape[0]} DRG families')

Payment Uplift Analysis
Average CC uplift: $3,257
Average MCC uplift: $11,706
Blended uplift (70% CC / 30% MCC): $5,791

Based on 313 DRG families


In [3]:
# Estimate shift rates based on documentation gap score
# Tiers aligned with composite Documentation Gap Score classification:
#   Low (0-49), Medium (50-64), High (65-100)
def estimate_shift_rate(score):
    """
    Maps documentation gap score to estimated severity shift rate.
    
    Tiers are aligned with the composite Documentation Gap Score tiers:
    - 0-49  (Low):    1-4% of cases could be shifted
    - 50-64 (Medium): 4-7% could be shifted
    - 65-100 (High):  7-15% could be shifted
    
    These rates are analyst-defined assumptions, NOT empirically calibrated.
    They are conservative vs. industry benchmarks (CDI programs typically
    improve 15-25% of targeted cases, but this model accounts for hospitals
    that may not have a full CDI program in place).
    
    Linear interpolation within each tier ensures smooth transitions
    and avoids hard jumps at tier boundaries.
    """
    if pd.isna(score):
        return 0.03  # Default to moderate shift rate
    
    if score < 50:
        # Low tier: linear interpolation from 1% at score=0 to 4% at score=49
        return 0.01 + (score / 49) * 0.03
    elif score < 65:
        # Medium tier: linear from 4% at score=50 to 7% at score=64
        return 0.04 + ((score - 50) / 14) * 0.03
    else:
        # High tier: linear from 7% at score=65 to 15% at score=100
        return 0.07 + ((score - 65) / 35) * 0.08

# Apply shift rate estimation
df['estimated_shift_rate'] = df['doc_gap_score'].apply(estimate_shift_rate)

# Display the mapping at key boundaries
print('Documentation Gap Score → Estimated Shift Rate')
print('=' * 55)
print(f'{"Score":>6}  {"Rate":>7}  {"Tier":<8}')
print('-' * 55)
test_scores = [0, 25, 49, 50, 57, 64, 65, 80, 100]
for score in test_scores:
    rate = estimate_shift_rate(score)
    if score < 50:
        tier = 'Low'
    elif score < 65:
        tier = 'Medium'
    else:
        tier = 'High'
    print(f'{score:6d}  {rate:6.2%}  {tier:<8}')

Documentation Gap Score → Estimated Shift Rate
 Score     Rate  Tier    
-------------------------------------------------------
     0   1.00%  Low     
    25   2.53%  Low     
    49   4.00%  Low     
    50   4.00%  Medium  
    57   5.50%  Medium  
    64   7.00%  Medium  
    65   7.00%  High    
    80  10.43%  High    
   100  15.00%  High    


In [4]:
# Calculate revenue impact per hospital
# Need discharge data — load from NB06 benchmarks since NB07 output doesn't include it

discharge_col = None

# Check if discharge column already exists in gap scores
for candidate in ['total_cc_discharges', 'medicare_discharges_puf', 'medicare_discharges_impact', 'total_discharges']:
    if candidate in df.columns and df[candidate].notna().any():
        discharge_col = candidate
        break

# If not found, load from NB06 benchmarks file
if discharge_col is None:
    print("No discharge column in gap scores — loading from NB06 benchmarks...")
    benchmarks_path = '../../data/outputs/nb06_peer_benchmarks/hospital_with_benchmarks.csv'
    df_bench = pd.read_csv(benchmarks_path, dtype={'ccn': str}, usecols=['ccn', 'medicare_discharges_impact', 'medicare_discharges_puf', 'total_cc_discharges'])
    df_bench['ccn'] = df_bench['ccn'].str.zfill(6)
    df['ccn'] = df['ccn'].str.zfill(6)
    df = df.merge(df_bench, on='ccn', how='left')
    
    for candidate in ['medicare_discharges_impact', 'medicare_discharges_puf', 'total_cc_discharges']:
        if candidate in df.columns and df[candidate].notna().any():
            discharge_col = candidate
            break

if discharge_col is None:
    raise ValueError('No discharge column found after loading benchmarks')

print(f'Using discharge column: {discharge_col}')
print(f'Hospitals with discharge data: {df[discharge_col].notna().sum():,}')

# Prepare discharge data (handle missing values)
df['shiftable_discharges'] = df[discharge_col].fillna(0).astype(float)

# Calculate shifted cases
df['estimated_shifted_cases'] = (
    df['shiftable_discharges'] * df['estimated_shift_rate']
).round(0).astype(int)

# Calculate annual revenue opportunity
df['estimated_annual_revenue_opportunity'] = (
    df['estimated_shifted_cases'] * blended_uplift
).round(0).astype(int)

# Summary statistics
print('\nRevenue Impact Summary')
print('='*50)
print(f'Total hospitals: {len(df):,}')
print(f'Hospitals with discharge data: {(df["shiftable_discharges"] > 0).sum():,}')
print(f'\nNational revenue opportunity: ${df["estimated_annual_revenue_opportunity"].sum():,.0f}')
print(f'Mean per hospital: ${df["estimated_annual_revenue_opportunity"].mean():,.0f}')
print(f'Median per hospital: ${df["estimated_annual_revenue_opportunity"].median():,.0f}')
print(f'Min: ${df["estimated_annual_revenue_opportunity"].min():,.0f}')
print(f'Max: ${df["estimated_annual_revenue_opportunity"].max():,.0f}')

No discharge column in gap scores — loading from NB06 benchmarks...
Using discharge column: medicare_discharges_impact
Hospitals with discharge data: 3,060

Revenue Impact Summary
Total hospitals: 3,280
Hospitals with discharge data: 3,060

National revenue opportunity: $1,895,939,768
Mean per hospital: $578,030
Median per hospital: $277,990
Min: $0
Max: $12,150,490


In [5]:
# Revenue impact by hospital segment
print('REVENUE OPPORTUNITY BY HOSPITAL SEGMENT')
print('='*70)

# By ownership category
if 'ownership_category' in df.columns:
    print('\n1. By Ownership Category')
    print('-'*70)
    ownership_summary = df.groupby('ownership_category').agg({
        'estimated_annual_revenue_opportunity': ['sum', 'mean', 'count']
    }).round(0)
    ownership_summary.columns = ['Total Opportunity', 'Mean per Hospital', 'Count']
    ownership_summary['Total Opportunity'] = ownership_summary['Total Opportunity'].apply(lambda x: f'${x:,.0f}')
    ownership_summary['Mean per Hospital'] = ownership_summary['Mean per Hospital'].apply(lambda x: f'${x:,.0f}')
    print(ownership_summary)

# By bed size tier
if 'bed_size_tier' in df.columns:
    print('\n2. By Bed Size Tier')
    print('-'*70)
    bed_size_order = ['Small', 'Medium', 'Large', 'Very Large']
    df_bed = df[df['bed_size_tier'].isin(bed_size_order)].copy()
    bed_summary = df_bed.groupby('bed_size_tier', observed=True).agg({
        'estimated_annual_revenue_opportunity': ['sum', 'mean', 'count']
    }).reindex(bed_size_order).round(0)
    bed_summary.columns = ['Total Opportunity', 'Mean per Hospital', 'Count']
    bed_summary['Total Opportunity'] = bed_summary['Total Opportunity'].apply(lambda x: f'${x:,.0f}')
    bed_summary['Mean per Hospital'] = bed_summary['Mean per Hospital'].apply(lambda x: f'${x:,.0f}')
    print(bed_summary)

# By gap tier
if 'gap_tier' in df.columns:
    print('\n3. By Documentation Gap Tier')
    print('-'*70)
    tier_order = ['Low', 'Medium', 'High']
    df_tier = df[df['gap_tier'].isin(tier_order)].copy()
    tier_summary = df_tier.groupby('gap_tier', observed=True).agg({
        'estimated_annual_revenue_opportunity': ['sum', 'mean', 'count']
    }).reindex(tier_order).round(0)
    tier_summary.columns = ['Total Opportunity', 'Mean per Hospital', 'Count']
    tier_summary['Total Opportunity'] = tier_summary['Total Opportunity'].apply(lambda x: f'${x:,.0f}')
    tier_summary['Mean per Hospital'] = tier_summary['Mean per Hospital'].apply(lambda x: f'${x:,.0f}')
    print(tier_summary)

# By census region
if 'census_region' in df.columns:
    print('\n4. By Census Region')
    print('-'*70)
    region_summary = df.groupby('census_region', observed=True).agg({
        'estimated_annual_revenue_opportunity': ['sum', 'mean', 'count']
    }).sort_values(('estimated_annual_revenue_opportunity', 'sum'), ascending=False).round(0)
    region_summary.columns = ['Total Opportunity', 'Mean per Hospital', 'Count']
    region_summary['Total Opportunity'] = region_summary['Total Opportunity'].apply(lambda x: f'${x:,.0f}')
    region_summary['Mean per Hospital'] = region_summary['Mean per Hospital'].apply(lambda x: f'${x:,.0f}')
    print(region_summary)

REVENUE OPPORTUNITY BY HOSPITAL SEGMENT

1. By Ownership Category
----------------------------------------------------------------------
                   Total Opportunity Mean per Hospital  Count
ownership_category                                           
For-Profit              $245,309,006          $376,819    651
Government              $222,913,420          $494,265    451
Nonprofit             $1,419,244,435          $730,064   1944
Other                     $8,472,907           $36,209    234

2. By Bed Size Tier
----------------------------------------------------------------------
              Total Opportunity Mean per Hospital  Count
bed_size_tier                                           
Small                      $nan              $nan    NaN
Medium                     $nan              $nan    NaN
Large                      $nan              $nan    NaN
Very Large                 $nan              $nan    NaN

3. By Documentation Gap Tier
---------------------------

In [6]:
# Top 25 hospitals by revenue opportunity
print('TOP 25 HOSPITALS BY REVENUE OPPORTUNITY')
print('='*100)

top_25 = df.nlargest(25, 'estimated_annual_revenue_opportunity')[[
    'ccn', 'hospital_name', 'state', 'beds', 'doc_gap_score',
    'estimated_shift_rate', 'estimated_shifted_cases', 'estimated_annual_revenue_opportunity'
]].copy()

print()
for idx, (i, row) in enumerate(top_25.iterrows(), 1):
    name = str(row["hospital_name"])[:35]
    beds = row["beds"]
    beds_str = f'{beds:.0f}' if pd.notna(beds) else 'N/A'
    print(f'{idx:2d}. {row["ccn"]} | {name:35s} | {row["state"]:2s} | '
          f'{beds_str:>5s} beds | Gap: {row["doc_gap_score"]:5.1f} | '
          f'Shift Rate: {row["estimated_shift_rate"]:6.2%} | '
          f'Shifted Cases: {row["estimated_shifted_cases"]:>8,d} | '
          f'Revenue: ${row["estimated_annual_revenue_opportunity"]:>12,d}')

TOP 25 HOSPITALS BY REVENUE OPPORTUNITY

 1. 100007 | ADVENTHEALTH ORLANDO                | FL |  3289 beds | Gap:  63.1 | Shift Rate:  6.81% | Shifted Cases:    2,098 | Revenue: $  12,150,490
 2. 080001 | CHRISTIANA HOSPITAL                 | DE |  1196 beds | Gap:  67.7 | Shift Rate:  7.62% | Shifted Cases:    1,348 | Revenue: $   7,806,893
 3. 450388 | METHODIST HOSPITAL                  | TX |  2465 beds | Gap:  64.8 | Shift Rate:  7.17% | Shifted Cases:    1,290 | Revenue: $   7,470,988
 4. 390133 | LEHIGH VALLEY HOSPITAL              | PA |  1190 beds | Gap:  68.7 | Shift Rate:  7.84% | Shifted Cases:    1,263 | Revenue: $   7,314,618
 5. 100088 | BAPTIST HEALTH MEDICAL CENTER - JAC | FL |  1229 beds | Gap:  71.2 | Shift Rate:  8.41% | Shifted Cases:    1,064 | Revenue: $   6,162,117
 6. 330101 | NEW YORK-PRESBYTERIAN HOSPITAL      | NY |  2262 beds | Gap:  46.7 | Shift Rate:  3.86% | Shifted Cases:    1,056 | Revenue: $   6,115,785
 7. 330214 | NYU LANGONE HOSPITALS             

## National Revenue Impact Summary

The documentation gap represents a significant missed revenue opportunity across U.S. hospitals. Better clinical documentation could enable hospitals to accurately reflect the severity of patient conditions and capture the appropriate reimbursement.

In [7]:
# National aggregate summary
total_opportunity = df['estimated_annual_revenue_opportunity'].sum()
high_gap_opportunity = df[df['gap_tier'] == 'High']['estimated_annual_revenue_opportunity'].sum() if 'gap_tier' in df.columns else 0
medium_gap_opportunity = df[df['gap_tier'] == 'Medium']['estimated_annual_revenue_opportunity'].sum() if 'gap_tier' in df.columns else 0
low_gap_opportunity = df[df['gap_tier'] == 'Low']['estimated_annual_revenue_opportunity'].sum() if 'gap_tier' in df.columns else 0

print('NATIONAL REVENUE IMPACT SUMMARY')
print('='*70)
print(f'\nTotal national documentation gap revenue opportunity:')
print(f'  ${total_opportunity:,.0f}')
print(f'\nBreakdown by gap tier:')
if 'gap_tier' in df.columns:
    print(f'  High-gap hospitals:   ${high_gap_opportunity:,.0f} ({high_gap_opportunity/total_opportunity*100:.1f}%)')
    print(f'  Medium-gap hospitals: ${medium_gap_opportunity:,.0f} ({medium_gap_opportunity/total_opportunity*100:.1f}%)')
    print(f'  Low-gap hospitals:    ${low_gap_opportunity:,.0f} ({low_gap_opportunity/total_opportunity*100:.1f}%)')

print(f'\nPer hospital metrics:')
print(f'  Average per hospital:  ${df["estimated_annual_revenue_opportunity"].mean():,.0f}')
print(f'  Median per hospital:   ${df["estimated_annual_revenue_opportunity"].median():,.0f}')
print(f'  Std deviation:         ${df["estimated_annual_revenue_opportunity"].std():,.0f}')

print(f'\nTotal hospitals analyzed: {len(df):,}')
print(f'Total discharges (shiftable): {df["shiftable_discharges"].sum():,.0f}')
print(f'Total estimated shifted cases: {df["estimated_shifted_cases"].sum():,.0f}')

# Calculate what this means per discharge
total_discharges = df['shiftable_discharges'].sum()
if total_discharges > 0:
    rev_per_discharge = total_opportunity / total_discharges
    print(f'\nRevenue per discharge (blended): ${rev_per_discharge:,.2f}')

NATIONAL REVENUE IMPACT SUMMARY

Total national documentation gap revenue opportunity:
  $1,895,939,768

Breakdown by gap tier:
  High-gap hospitals:   $239,112,146 (12.6%)
  Medium-gap hospitals: $1,104,849,056 (58.3%)
  Low-gap hospitals:    $551,978,566 (29.1%)

Per hospital metrics:
  Average per hospital:  $578,030
  Median per hospital:   $277,990
  Std deviation:         $814,190

Total hospitals analyzed: 3,280
Total discharges (shiftable): 6,669,632
Total estimated shifted cases: 327,368

Revenue per discharge (blended): $284.26


In [8]:
# Prepare output dataset
output_cols = [
    'ccn', 'hospital_name', 'state', 'beds', 'ownership_category',
    'doc_gap_score', 'gap_tier', 'census_region', 'bed_size_tier',
    'shiftable_discharges', 'estimated_shift_rate',
    'estimated_shifted_cases', 'estimated_annual_revenue_opportunity'
]

# Filter to only columns that exist
output_cols = [col for col in output_cols if col in df.columns]

df_output = df[output_cols].copy()

# Save to CSV
import os
output_dir = os.path.dirname(output_path)
os.makedirs(output_dir, exist_ok=True)

df_output.to_csv(output_path, index=False)

print('Output saved successfully!')
print(f'\nFile: {output_path}')
print(f'Rows: {len(df_output):,}')
print(f'Columns: {len(df_output.columns)}')
print(f'\nOutput columns:')
for col in df_output.columns:
    print(f'  - {col}')

Output saved successfully!

File: ../../data/outputs/nb08_revenue_impact/hospital_revenue_impact.csv
Rows: 3,280
Columns: 13

Output columns:
  - ccn
  - hospital_name
  - state
  - beds
  - ownership_category
  - doc_gap_score
  - gap_tier
  - census_region
  - bed_size_tier
  - shiftable_discharges
  - estimated_shift_rate
  - estimated_shifted_cases
  - estimated_annual_revenue_opportunity
